In [ ]:
import os
from datetime import datetime
import openpyxl
import pandas as pd

# ==============================================================================
# 0. KLASÖR VE DOSYA YOLLARI (Merkezi Tanımlama)
# ==============================================================================
girdi_klasoru = r""
ana_cikti_klasoru = r""

# Ara ve Nihai Dosya Yolları
dosya_1_temiz = os.path.join(ana_cikti_klasoru, "Tum_Bolgeler_Veri_Temiz.xlsx")
dosya_2_rasat_value = os.path.join(ana_cikti_klasoru, "rasat_value.xlsx")
dosya_3_fix_value = os.path.join(ana_cikti_klasoru, "fix_rasat_value.xlsx")
dosya_4_negatif = os.path.join(ana_cikti_klasoru, "rasat_value_negatif.xlsx")
dosya_5_id_value = os.path.join(ana_cikti_klasoru, "ID_rasat_value.xlsx")

# Collar giriş ve çıkış dosyaları
girdi_collar_ham = r""
dosya_6_id_collar = os.path.join(ana_cikti_klasoru, "ID_rasat_collar.xlsx")

if ana_cikti_klasoru and not os.path.exists(ana_cikti_klasoru):
    os.makedirs(ana_cikti_klasoru)


# ==============================================================================
# 1. ADIM: HAM EXCEL DOSYALARINDAN ZAMAN SERİSİ OLUŞTURMA
# ==============================================================================
izin_verilen_kuyular = {
    # "SayfaAdi": ["KuyuNo1", "KuyuNo2"]
}

ay_isimleri = {
    'ocak': 1, 'şubat': 2, 'subat': 2, 'mart': 3, 'nisan': 4, 'mayıs': 5, 'mayis': 5, 'haziran': 6,
    'temmuz': 7, 'ağustos': 8, 'agustos': 8, 'eylül': 9, 'eylul': 9, 'ekim': 10, 'kasım': 11, 'kasim': 11,
    'aralık': 12, 'aralik': 12
}

hedef_basliklar = ["SayfaAdı", "KuyuNo", "Tarih(Yıl-Ay)", "OlcumDegeri"]

wb_hedef = openpyxl.Workbook()
hedef_sayfa = wb_hedef.active
hedef_sayfa.title = 'Rasat_Value'

for idx, baslik in enumerate(hedef_basliklar, start=1):
    hedef_sayfa.cell(row=1, column=idx, value=baslik)

hedef_satir = 2

excel_dosyalari = [f for f in os.listdir(girdi_klasoru) if f.endswith('.xlsx') and not f.startswith('~$')]
print(f"[1. ADIM] Toplam {len(excel_dosyalari)} adet Excel dosyası işleniyor...\n")

for dosya_adi in excel_dosyalari:
    dosya_yolu = os.path.join(girdi_klasoru, dosya_adi)
    print(f"-> {dosya_adi} taranıyor...")

    wb = openpyxl.load_workbook(dosya_yolu, data_only=True)

    for sayfa_adi in wb.sheetnames:
        if sayfa_adi not in izin_verilen_kuyular:
            continue

        gecerli_kuyular = izin_verilen_kuyular[sayfa_adi]
        sheet = wb[sayfa_adi]

        # A. Kuyu No satırını bulma
        kuyu_satiri = -1
        for r in range(1, 20):
            for c in range(1, 10):
                val = sheet.cell(row=r, column=c).value
                if val is not None:
                    s_val = str(val).lower().replace(" ", "").replace("\xa0", "").replace("\n", "")
                    if "kuyuno" in s_val or s_val == "kuyu":
                        kuyu_satiri = r
                        break
            if kuyu_satiri != -1:
                break

        # B. Zaman serisi başlangıç satırını bulma
        yil_satiri, yil_sutunu, ay_sutunu = -1, -1, -1
        for r in range(1, 35):
            for c in range(1, 5):
                cell_value = sheet.cell(row=r, column=c).value
                if cell_value is not None:
                    val = str(cell_value).strip().lower()
                    if val in ['yil', 'yıl', 'yıllar']:
                        yil_satiri = r
                        yil_sutunu = c
                        break
            if yil_satiri != -1:
                break

        if yil_satiri != -1:
            for ac in range(yil_sutunu + 1, yil_sutunu + 4):
                cell_value = sheet.cell(row=yil_satiri, column=ac).value
                if cell_value is not None:
                    aval = str(cell_value).strip().lower()
                    if aval in ['ay', 'aylar']:
                        ay_sutunu = ac
                        break
            if ay_sutunu == -1:
                ay_sutunu = yil_sutunu + 1

        # C. Kuyuları ve verileri hafızaya (sözlüğe) alma
        if yil_satiri != -1:
            max_row = sheet.max_row
            max_col = sheet.max_column
            hedef_kuyu_satiri = kuyu_satiri if kuyu_satiri != -1 else 1

            kuyular = {}
            for c in range(5, max_col + 1):
                k_val = sheet.cell(row=hedef_kuyu_satiri, column=c).value
                if k_val is not None and str(k_val).strip() != "":
                    kuyu_ismi = str(k_val).strip()
                    if kuyu_ismi in gecerli_kuyular:
                        kuyular[c] = kuyu_ismi

            if not kuyular:
                continue

            olcum_verileri = {c: {} for c in kuyular.keys()}
            max_yil = 1966
            mevcut_yil = None

            for r in range(yil_satiri + 1, max_row + 1):
                y_val = sheet.cell(row=r, column=yil_sutunu).value
                if y_val is not None:
                    y_str = str(y_val).strip()
                    if y_str.lower() not in ['nan', '', 'none']:
                        try:
                            y_int = int(float(y_str))
                            if 1900 < y_int < 2100:
                                mevcut_yil = y_int
                                if mevcut_yil > max_yil:
                                    max_yil = mevcut_yil
                        except (ValueError, TypeError):
                            pass

                if mevcut_yil is not None:
                    m_val = sheet.cell(row=r, column=ay_sutunu).value
                    m_str = str(m_val).strip().lower() if m_val is not None else ""
                    ay = None

                    if m_str in ay_isimleri:
                        ay = ay_isimleri[m_str]
                    else:
                        try:
                            ay = int(float(m_str))
                        except (ValueError, TypeError):
                            pass

                    if ay is not None and 1 <= ay <= 12:
                        for c in kuyular.keys():
                            val = sheet.cell(row=r, column=c).value
                            if val is not None and str(val).strip().lower() not in ['', 'nan', 'none']:
                                olcum_verileri[c][(mevcut_yil, ay)] = val

            # D. Hedef dosyaya 1966'dan itibaren yazma
            for c, kuyu_no in kuyular.items():
                for y in range(1966, max_yil + 1):
                    for m in range(1, 13):
                        deger = olcum_verileri[c].get((y, m), "null")

                        hedef_sayfa.cell(row=hedef_satir, column=1, value=sayfa_adi)
                        hedef_sayfa.cell(row=hedef_satir, column=2, value=kuyu_no)

                        tarih_obj = datetime(y, m, 1)
                        tarih_hucresi = hedef_sayfa.cell(row=hedef_satir, column=3, value=tarih_obj)
                        tarih_hucresi.number_format = 'YYYY-MM'

                        hedef_sayfa.cell(row=hedef_satir, column=4, value=deger)
                        hedef_satir += 1

    wb.close()

wb_hedef.save(dosya_1_temiz)
print(f"-> 1. Adım Tamamlandı: {hedef_satir - 2} satır oluşturuldu ({dosya_1_temiz})\n")


# ==============================================================================
# 2. ADIM: "KOT" İÇEREN SAYFALARIN POZİTİF DEĞERLERİNİ 5. SÜTUNA KOPYALAMA
# ==============================================================================
print("[2. ADIM] 'Kot' içeren sayfaların değerleri 5. sütuna kopyalanıyor...")
wb = openpyxl.load_workbook(dosya_1_temiz)
sheet = wb.active

for r in range(1, sheet.max_row + 1):
    hucre_A = sheet.cell(row=r, column=1).value  # A sütunu (SayfaAdı)
    hucre_D = sheet.cell(row=r, column=4).value  # D sütunu (OlcumDegeri)

    if hucre_A is not None and "kot" in str(hucre_A).lower():
        if hucre_D is not None:
            try:
                deger = float(hucre_D)
                if deger > 0:
                    sheet.cell(row=r, column=5).value = deger
            except (ValueError, TypeError):
                pass

wb.save(dosya_2_rasat_value)
print(f"-> 2. Adım Tamamlandı: {dosya_2_rasat_value}\n")


# ==============================================================================
# 3. ADIM: COLLAR DOSYASINDAKİ ZEMİN KOTU İLE FARK HESAPLAMA
# ==============================================================================
print("[3. ADIM] Collar dosyasındaki 7. sütun değerleri ile çıkarma işlemi yapılıyor...")
wb_collar = openpyxl.load_workbook(girdi_collar_ham, data_only=True)
sheet_collar = wb_collar.active

collar_verileri = {}
for r in range(2, sheet_collar.max_row + 1):
    id_hucre = sheet_collar.cell(row=r, column=2).value  # B Sütunu (Kuyu No)
    deger_7 = sheet_collar.cell(row=r, column=7).value   # 7. Sütun (Zemin Kotu / Çıkarılacak Değer)

    if id_hucre is not None and deger_7 is not None:
        kuyu_no = str(id_hucre).strip()
        try:
            collar_verileri[kuyu_no] = float(deger_7)
        except (ValueError, TypeError):
            continue

wb_collar.close()

wb_value = openpyxl.load_workbook(dosya_2_rasat_value)
sheet_value = wb_value.active
islem_sayisi = 0

for r in range(2, sheet_value.max_row + 1):
    id_hucre = sheet_value.cell(row=r, column=2).value  # B Sütunu (Kuyu No)
    deger_5 = sheet_value.cell(row=r, column=5).value   # E Sütunu (5. Sütun)

    if id_hucre is not None and deger_5 is not None:
        kuyu_no = str(id_hucre).strip()
        if kuyu_no in collar_verileri:
            try:
                sayi_5 = float(deger_5)
                cikarilacak_sayi = collar_verileri[kuyu_no]
                sonuc = -(cikarilacak_sayi - sayi_5)

                sheet_value.cell(row=r, column=5).value = sonuc
                islem_sayisi += 1
            except (ValueError, TypeError):
                continue

wb_value.save(dosya_3_fix_value)
print(f"-> 3. Adım Tamamlandı: Toplam {islem_sayisi} satır hesaplandı ({dosya_3_fix_value})\n")


# ==============================================================================
# 4. ADIM: 4. SÜTUNDAKİ POZİTİF DEĞERLERİ NEGATİFE ÇEVİRME
# ==============================================================================
print("[4. ADIM] 4. sütundaki pozitif ölçüm değerleri negatife çevriliyor...")
wb = openpyxl.load_workbook(dosya_3_fix_value)
sheet = wb.active
degisen_sayisi = 0

for r in range(1, sheet.max_row + 1):
    hucre_D = sheet.cell(row=r, column=4).value

    if hucre_D is not None:
        try:
            deger = float(hucre_D)
            if deger > 0:
                yeni_deger = deger * -1
                if isinstance(hucre_D, int) or (isinstance(hucre_D, float) and hucre_D.is_integer()):
                    yeni_deger = int(yeni_deger)

                sheet.cell(row=r, column=4).value = yeni_deger
                degisen_sayisi += 1
        except (ValueError, TypeError):
            pass

wb.save(dosya_4_negatif)
print(f"-> 4. Adım Tamamlandı: {degisen_sayisi} adet pozitif değer negatife çevrildi ({dosya_4_negatif})\n")


# ==============================================================================
# 5. ADIM: RASAT_VALUE VE RASAT_COLLAR DOSYALARINA 1000'DEN BAŞLAYAN ID ATAMA
# ==============================================================================
print("[5. ADIM] Kuyulara 1000'den başlayan tekil ID numaraları atanıyor...")

# 5A. rasat_value dosyasına ID atama ve sözlüğü oluşturma
wb_val = openpyxl.load_workbook(dosya_4_negatif)
sheet_val = wb_val.active

kaynak_sutun = 2
hedef_sutun_val = 5
baslangic_id = 1000
kuyu_id_sozlugu = {}

for r in range(2, sheet_val.max_row + 1):
    hucre_degeri = sheet_val.cell(row=r, column=kaynak_sutun).value
    if hucre_degeri is not None:
        aranan_deger = str(hucre_degeri).strip()
        if aranan_deger not in kuyu_id_sozlugu:
            kuyu_id_sozlugu[aranan_deger] = baslangic_id
            baslangic_id += 1

        sheet_val.cell(row=r, column=hedef_sutun_val).value = kuyu_id_sozlugu[aranan_deger]

wb_val.save(dosya_5_id_value)
print(f"-> Value ID ataması tamamlandı ({len(kuyu_id_sozlugu)} tekil kuyu): {dosya_5_id_value}")

# 5B. Oluşturulan kuyu_id_sozlugu ile rasat_collar dosyasına ID atama
wb_col = openpyxl.load_workbook(girdi_collar_ham)
sheet_col = wb_col.active
hedef_sutun_col = 20

for r in range(2, sheet_col.max_row + 1):
    hucre_degeri = sheet_col.cell(row=r, column=kaynak_sutun).value
    if hucre_degeri is not None:
        aranan_deger = str(hucre_degeri).strip()
        if aranan_deger in kuyu_id_sozlugu:
            sheet_col.cell(row=r, column=hedef_sutun_col).value = kuyu_id_sozlugu[aranan_deger]
        else:
            sheet_col.cell(row=r, column=hedef_sutun_col).value = "Listede Yok"

wb_col.save(dosya_6_id_collar)
print(f"-> Collar ID ataması tamamlandı: {dosya_6_id_collar}\n")


# ==============================================================================
# 6. ADIM: İKİ DOSYA ARASINDAKİ KUYU NUMARALARINI KARŞILAŞTIRMA (DOĞRULAMA)
# ==============================================================================
print("[6. ADIM] ID_rasat_value ve ID_rasat_collar dosyaları karşılaştırılıyor...")
df_value = pd.read_excel(dosya_5_id_value, usecols=[1])
df_collar = pd.read_excel(dosya_6_id_collar, usecols=[1])

sutun_value = df_value.columns[0]
sutun_collar = df_collar.columns[0]

value_kumesi = set(df_value[sutun_value].dropna().astype(str).str.strip())
collar_kumesi = set(df_collar[sutun_collar].dropna().astype(str).str.strip())

sadece_value_da_olanlar = value_kumesi - collar_kumesi
sadece_collar_da_olanlar = collar_kumesi - value_kumesi

print("\n--- KARŞILAŞTIRMA SONUCU ---")
print(f"rasat_value (B Sütunu) eşsiz değer sayısı: {len(value_kumesi)}")
print(f"rasat_collar (B Sütunu) eşsiz değer sayısı: {len(collar_kumesi)}\n")

if not sadece_value_da_olanlar and not sadece_collar_da_olanlar:
    print("Mükemmel! Her iki dosyadaki veriler birebir aynı.")
else:
    if sadece_value_da_olanlar:
        print(f"Sadece 'rasat_value' içinde olanlar ({len(sadece_value_da_olanlar)} adet):")
        for val in list(sadece_value_da_olanlar)[:20]:
            print(f"  - {val}")
    if sadece_collar_da_olanlar:
        print(f"Sadece 'rasat_collar' içinde olanlar ({len(sadece_collar_da_olanlar)} adet):")
        for val in list(sadece_collar_da_olanlar)[:20]:
            print(f"  - {val}")